#### Create table by using spark SQL

In [0]:
spark.sql("DROP TABLE IF EXISTS employee")
spark.sql("DROP TABLE IF EXISTS emp")
spark.sql("DROP TABLE IF EXISTS employees")

DataFrame[]

In [0]:
%sql
CREATE TABLE EMPLOYEE1(ID INT, NAME VARCHAR(40), SALARY FLOAT);
INSERT INTO EMPLOYEE1 VALUES(1,'jOHN', 40000), (2,'DAVIS', 50000);
SELECT * FROM EMPLOYEE1;

ID,NAME,SALARY
1,jOHN,40000.0
2,DAVIS,50000.0


#### 11.1- Creating data frame
* Dataframe is in Github

In [0]:
# Sample data
data = [
    (1, "John", "Doe", "Engineering", 70000),
    (2, "Jane", "Smith", "HR", 50000),
    (3, "Mike", "Johnson", "Finance", 60000),
    (4, "Emily", "Davis", "Engineering", 75000),
    (5, "Robert", "Brown", "Marketing", 55000),
    (6, "Linda", "Wilson", "HR", 52000)
]

# Column names
columns = ["ID", "FIRST_NAME", "LAST_NAME", "DEPARTMENT", "SALARY"]

# Create DataFrame
df_dept = spark.createDataFrame(data, columns)

# Show the DataFrame
display(df_dept)

ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
1,John,Doe,Engineering,70000
2,Jane,Smith,HR,50000
3,Mike,Johnson,Finance,60000
4,Emily,Davis,Engineering,75000
5,Robert,Brown,Marketing,55000
6,Linda,Wilson,HR,52000


#### 11.2-Creating temporary views on DataFrames.
* Temporary views let you run SQL queries on DataFrames like database tables.

In [0]:
# Register DataFrame as a temporary view
df_dept.createOrReplaceTempView("employees")

#### 11.2.1 Perform Simple SQL Transformations using SQL

In [0]:
%sql
SELECT * FROM employees WHERE DEPARTMENT = 'Engineering';


ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
1,John,Doe,Engineering,70000
4,Emily,Davis,Engineering,75000


In [0]:
# pyspark code by using .filter
df_dept.filter(df_dept["DEPARTMENT"] == "Engineering").show()

# pyspark code by using .where
df_dept.where("DEPARTMENT = 'Engineering'").show()


+---+----------+---------+-----------+------+
| ID|FIRST_NAME|LAST_NAME| DEPARTMENT|SALARY|
+---+----------+---------+-----------+------+
|  1|      John|      Doe|Engineering| 70000|
|  4|     Emily|    Davis|Engineering| 75000|
+---+----------+---------+-----------+------+

+---+----------+---------+-----------+------+
| ID|FIRST_NAME|LAST_NAME| DEPARTMENT|SALARY|
+---+----------+---------+-----------+------+
|  1|      John|      Doe|Engineering| 70000|
|  4|     Emily|    Davis|Engineering| 75000|
+---+----------+---------+-----------+------+



#### 11.2.2- We can perform transformation using spark sql
* Select all HR employees

In [0]:
spark.sql("SELECT * FROM employees WHERE DEPARTMENT = 'HR'").show()

+---+----------+---------+----------+------+
| ID|FIRST_NAME|LAST_NAME|DEPARTMENT|SALARY|
+---+----------+---------+----------+------+
|  2|      Jane|    Smith|        HR| 50000|
|  6|     Linda|   Wilson|        HR| 52000|
+---+----------+---------+----------+------+



#### 11.2.3- Select employees with salary > 60000 using SQL

In [0]:
%sql
SELECT FIRST_NAME, LAST_NAME, SALARY FROM employees WHERE SALARY > 60000;




FIRST_NAME,LAST_NAME,SALARY
John,Doe,70000
Emily,Davis,75000


In [0]:
#using pyspark
df_dept.filter(df_dept["SALARY"] > 60000).select("FIRST_NAME", "LAST_NAME", "SALARY").show()


+----------+---------+------+
|FIRST_NAME|LAST_NAME|SALARY|
+----------+---------+------+
|      John|      Doe| 70000|
|     Emily|    Davis| 75000|
+----------+---------+------+




#### 11.2.3- Select employees with salary > 60000 using spark sql

In [0]:
spark.sql("SELECT FIRST_NAME, LAST_NAME, SALARY FROM employees WHERE SALARY > 60000").show()


+----------+---------+------+
|FIRST_NAME|LAST_NAME|SALARY|
+----------+---------+------+
|      John|      Doe| 70000|
|     Emily|    Davis| 75000|
+----------+---------+------+



#### 11.2.4- Get average salary per department using pyspark sql
* In Python (and therefore PySpark), the triple quotes """ or ''' are used to define multi-line strings. 
* This is especially useful when writing SQL queries that span multiple lines, which improves readability and organization.
* Double quotes (" ") = single-line string

* Triple quotes (""" """ or ''' ''') = multi-line string

In [0]:
spark.sql("""
    SELECT DEPARTMENT, AVG(SALARY) AS AVG_SALARY
    FROM employees
    GROUP BY DEPARTMENT
""").show()


+-----------+----------+
| DEPARTMENT|AVG_SALARY|
+-----------+----------+
|Engineering|   72500.0|
|         HR|   51000.0|
|    Finance|   60000.0|
|  Marketing|   55000.0|
+-----------+----------+



##### 11.2.5- Without Triple Quotes (Less Readable):

In [0]:
spark.sql("SELECT DEPARTMENT, AVG(SALARY) AS AVG_SALARY FROM employees GROUP BY DEPARTMENT").show()


+-----------+----------+
| DEPARTMENT|AVG_SALARY|
+-----------+----------+
|Engineering|   72500.0|
|         HR|   51000.0|
|    Finance|   60000.0|
|  Marketing|   55000.0|
+-----------+----------+



#### 11.4-Global temporary views:

##### 11.4.1- Creating a Global Temporary View

In [0]:
df_dept.createOrReplaceTempView("employees_view")

# This registers the view as employees_view

##### 11.4.2- Accessing a Global Temporary View

In [0]:
# Using spark.sql()
spark.sql("SELECT * FROM employees_view").show()

# Using %sql in notebook
display(spark.sql("SELECT * FROM employees_view"))


+---+----------+---------+-----------+------+
| ID|FIRST_NAME|LAST_NAME| DEPARTMENT|SALARY|
+---+----------+---------+-----------+------+
|  1|      John|      Doe|Engineering| 70000|
|  2|      Jane|    Smith|         HR| 50000|
|  3|      Mike|  Johnson|    Finance| 60000|
|  4|     Emily|    Davis|Engineering| 75000|
|  5|    Robert|    Brown|  Marketing| 55000|
|  6|     Linda|   Wilson|         HR| 52000|
+---+----------+---------+-----------+------+



ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
1,John,Doe,Engineering,70000
2,Jane,Smith,HR,50000
3,Mike,Johnson,Finance,60000
4,Emily,Davis,Engineering,75000
5,Robert,Brown,Marketing,55000
6,Linda,Wilson,HR,52000


##### 11.4.3-Why Use Global Temporary Views?
* **Useful when:**

* You want the view to be available in multiple notebooks

* You want to share a view across different sessions/users on the same Spark application

#### 11.4.4- Accessing temp view and global temp view from different notebook

In [0]:
%sql
SELECT * FROM employees; -- Temp view

SELECT * FROM employees_view; -- Global temp view

ID,FIRST_NAME,LAST_NAME,DEPARTMENT,SALARY
1,John,Doe,Engineering,70000
2,Jane,Smith,HR,50000
3,Mike,Johnson,Finance,60000
4,Emily,Davis,Engineering,75000
5,Robert,Brown,Marketing,55000
6,Linda,Wilson,HR,52000


##### 11.4.5- Command to check 	global_temp database and temp view database

In [0]:
# Check tables Using SQL command 
spark.sql("SHOW TABLES IN default").show()

# OR
# Using pyspark 
spark.catalog.listTables("default")

# Check temp_table Using Pyspark command, unfortunately we dont have command to check temp table using sql command 
spark.catalog.listTables()

+--------+--------------+-----------+
|database|     tableName|isTemporary|
+--------+--------------+-----------+
| default|     employee1|      false|
|        |     employees|       true|
|        |employees_view|       true|
+--------+--------------+-----------+



[Table(name='employee1', catalog='workspace', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='employees', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employees_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

#### 11.6-Creating Spark SQL tables from DataFrame and vice versa

##### 11.6.1-Create a Spark SQL Table from a DataFrame
* You already have a DataFrame (in memory), and you want to save it as a permanent table in the metastore so it can be queried with SQL anytime.

In [0]:
# Sample DataFrame
data = [
    (1, "John", 70000),
    (2, "Jane", 50000)
]
columns = ["ID", "NAME", "SALARY"]
df = spark.createDataFrame(data, columns)

# Save as a table (permanent table in metastore)
df.write.mode("overwrite").saveAsTable("employee_table")

# It saves the DataFrame as a physical table in the Hive metastore (or Unity Catalog).

In [0]:
%sql
--Query the table
SELECT * FROM employee_table;
DESCRIBE DETAIL employee_table;
DESCRIBE employee_table;

col_name,data_type,comment
ID,bigint,null
NAME,string,null
SALARY,bigint,null


In [0]:
%sql
DESCRIBE EXTENDED employee_table;

col_name,data_type,comment
ID,bigint,null
NAME,string,null
SALARY,bigint,null
,,
# Delta Statistics Columns,,
Column Names,"ID, NAME, SALARY",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,


In [0]:
%sql
DESCRIBE DETAIL employee_table;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e2a86776-2453-46ff-a494-fc5081444bd0,workspace.default.employee_table,null,,2026-10-09T06:21:30.073Z,2026-10-09T06:21:33.000Z,List(),List(),1,1283,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


##### 11.6.2- Create a DataFrame from a Spark SQL Table

In [0]:
df_employee = spark.table("employee_table")
df_employee.show()

# It reads the data from the table employee_table into a DataFrame again.

# You can now apply transformations using the DataFrame API.

+---+----+------+
| ID|NAME|SALARY|
+---+----+------+
|  1|John| 70000|
|  2|Jane| 50000|
+---+----+------+



##### 11.6.3- Create Table Using SQL Directly

In [0]:
%sql
CREATE TABLE employee_sql (
    ID INT,
    NAME STRING,
    SALARY INT
);

INSERT INTO employee_sql VALUES (3, 'Robert', 60000);
SELECT * FROM employee_sql;

ID,NAME,SALARY
3,Robert,60000


##### 11.6.4- Create a DataFrame from SQL Table

In [0]:
df3 = spark.table("employee_sql")
df3.show()

+---+------+------+
| ID|  NAME|SALARY|
+---+------+------+
|  3|Robert| 60000|
+---+------+------+



#### 11.8-Types of tables:
* Managed table
* External table
* Managed table vs external table


##### 11.8.1- How to Create Managed table

In [0]:
%sql
-- 1- By using SQL
CREATE OR REPLACE TABLE managed_employee (
    ID INT,
    NAME STRING,
    SALARY INT
);

INSERT INTO managed_employee VALUES (1, 'John', 70000);
SELECT * FROM managed_employee;



 

ID,NAME,SALARY
1,John,70000


In [0]:
# OR

# 2- By using python
df.write.mode("overwrite").saveAsTable("managed_employee")

# Access table using python
spark.table("managed_employee").show()

+---+----+------+
| ID|NAME|SALARY|
+---+----+------+
|  1|John| 70000|
|  2|Jane| 50000|
+---+----+------+



In [0]:
%sql      --Create managed table
CREATE TABLE IF NOT EXISTS managed_employee (
    ID INT,
    NAME STRING,
    SALARY INT
);

INSERT INTO managed_employee VALUES (1, 'John', 70000);
SELECT * FROM managed_employee;

ID,NAME,SALARY
1,John,70000
2,Jane,50000
1,John,70000


##### 11.8.2- Check the table format using DESCRIBE FORMATTED

In [0]:
%sql --- Check the table format using DESCRIBE FORMATTED
DESCRIBE FORMATTED managed_employee;

col_name,data_type,comment
ID,int,null
NAME,string,null
SALARY,int,null
,,
# Delta Statistics Columns,,
Column Names,"ID, NAME, SALARY",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,


##### 11.8.3- How to Create External table

In [0]:
# Create table from CSV data
# On serverless compute, /mnt/ paths are not accessible and external tables
# require cloud storage URIs (e.g., s3://). Using a UC Volume as the data source instead.

# Write sample CSV data to a UC Volume
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.employee_data")
csv_data = "ID,NAME,SALARY\n1,John,70000\n2,Jane,50000\n"
dbutils.fs.put("/Volumes/workspace/default/employee_data/employees/data.csv", csv_data, overwrite=True)

# Read CSV from the volume and save as a table
spark.sql("DROP TABLE IF EXISTS external_employee")
df = spark.read.csv("/Volumes/workspace/default/employee_data/employees/", header=True, schema="ID INT, NAME STRING, SALARY INT")
df.write.mode("overwrite").saveAsTable("external_employee")

# Access the table using SQL
spark.sql("SELECT * FROM external_employee").show()

# OR
# Access using Python
spark.table("external_employee").show()


Wrote 41 bytes.
+---+----+------+
| ID|NAME|SALARY|
+---+----+------+
|  1|John| 70000|
|  2|Jane| 50000|
+---+----+------+

+---+----+------+
| ID|NAME|SALARY|
+---+----+------+
|  1|John| 70000|
|  2|Jane| 50000|
+---+----+------+



##### 11.8.4- Delete managed table

In [0]:
%sql
DROP TABLE managed_employee;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6161560830243015>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DROP TABLE managed_employee;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_logger.logExecut

#### 11.9-Use cases of DataFrames & SparkSQL

* DataFrame: A table-like data structure in Spark (similar to Excel or SQL table)

* Spark SQL: Allows you to run SQL queries on your data, just like querying a database

##### 11.9.1- Filtering and Analyzing Data
**Using DataFrame API:**
* df.filter(df.SALARY > 60000).show()

**Using Spark SQL**
* SELECT * FROM employees WHERE SALARY > 60000;

##### 11.9.2- Aggregating Data (e.g., Group By, Average)

**Using DataFrame API:**
* df.groupBy("DEPARTMENT").avg("SALARY").show()

**Using Spark SQL**
* SELECT DEPARTMENT, AVG(SALARY) AS AVG_SALARY
* FROM employees
* GROUP BY DEPARTMENT;

##### 11.9.3- Joining Tables

**Using DataFrame API:**
* df1.join(df2, df1.DEPT_ID == df2.DEPT_ID).show()

**Using Spark SQL**
* SELECT e.NAME, d.DEPT_NAME
* FROM employees e
* JOIN departments d ON e.DEPT_ID = d.DEPT_ID;